# Import

In [1]:
import os
import glob
import numpy as np
import graphical_sampling as gs
import pandas as pd

# NHT Variance

In [27]:
N = 100
n = 4
num_zones = 4
rng = gs.random.rng()
coords = rng.rand_coord((N, 2))
variable = np.random.rand(N) * 10
# inclusions = rng.unequal_inclusions(n, N)
inclusions = variable + np.random.rand(N)
pop = gs.Population(coords, inclusions, n=n, variable=variable)

In [2]:
df = pd.read_csv('populations/MU284_filtered.csv')
cs82 = df['CS82'].values.astype(float)
rmt85 = df['RMT85'].values.astype(float)
ss82 = df['SS82'].values.astype(float)

N = len(df)
n = 10
num_zones = 4
rng = gs.random.rng()
coords = rng.rand_coord((N, 2))
pop = gs.Population(
    coords=rng.rand_coord((N, 2)),
    inclusions=ss82,
    variable=cs82,
    n=4
)

In [3]:
initial_designs = [gs.Design(pop, num_zones=num_zones) for _ in range(100)]
criteria = gs.criteria.VarNHT()
gbfs = gs.search.GreedyBestFirstSearch(initial_designs, criteria)

In [4]:
gbfs.run(
    max_iterations=1000,
    max_open_set_size=2000,
    top_k=10,
    num_new_order_nodes=0,
    num_new_exchange_nodes=5,
    num_clusters=2,
    num_zones=1,
    num_changes=2,
    num_zone_changes=1,
    random_pull=True,
    exchange_coef=1.0,
    num_explore=5
)

--- Starting Parallel GBFS: Max Iterations=1000, Batch Size=5, Workers=-1 ---
Initial best criteria value: 498813.2237
Iter     0/1000 | Best: 498813.2237 | Open:   100 | Closed:     0
  [!] New best found at iter 0: 493522.1303 (improved by 5291.0934)
  [!] New best found at iter 1: 488712.5256 (improved by 4809.6047)
  [!] New best found at iter 2: 484458.5789 (improved by 4253.9467)
  [!] New best found at iter 2: 483865.2656 (improved by 593.3133)
  [!] New best found at iter 2: 477902.3794 (improved by 5962.8862)
  [!] New best found at iter 3: 477298.6168 (improved by 603.7626)
  [!] New best found at iter 4: 471269.6208 (improved by 6028.9959)
  [!] New best found at iter 4: 470507.9331 (improved by 761.6878)
  [!] New best found at iter 5: 470199.7306 (improved by 308.2025)
  [!] New best found at iter 5: 466570.5962 (improved by 3629.1344)
  [!] New best found at iter 6: 466013.4427 (improved by 557.1535)
  [!] New best found at iter 7: 465103.5046 (improved by 909.9381)
  [!]

# Spread

In [5]:
N = 100
n = 4
num_zones = 4
rng = gs.random.rng()
coords = rng.rand_coord((N, 2))
variable = np.random.rand(N) * 10
# inclusions = rng.unequal_inclusions(n, N)
inclusions = variable + np.random.rand(N)
pop = gs.Population(coords, inclusions, n=n, variable=variable)

In [6]:
DATA_DIR = "populations"
csv_paths = glob.glob(os.path.join(DATA_DIR, "*.csv"))

coords_dict = {}
probs_dict = {}

for fp in csv_paths:
    name = os.path.splitext(os.path.basename(fp))[0]
    data = np.loadtxt(fp, delimiter=",", skiprows=1)
    coords = data[:, :2]
    probs  = data[:, -1]

    coord_name, prob_name, *rest = name.split("_")
    coord_name = 'cluster' if coord_name == 'clust' else coord_name
    prob_name = 'equal' if prob_name == 'eq' else 'unequal'

    coords_dict[coord_name] = coords
    probs_dict[coord_name] = probs_dict.get(coord_name, {})
    probs_dict[coord_name][prob_name] = probs

print(coords_dict.keys())

dict_keys(['swiss', 'RegularPop1000', 'AggregatedPop1027', 'cluster', 'meuse', 'random', 'grid', 'MU284'])


In [7]:
N = 100
n = 5
coords = coords_dict['random']
probs = probs_dict['random']['unequal']
pop = gs.Population(coords, probs, n=n)

In [11]:
initial_designs = []
for _ in range(20):
    fbn = gs.clustering.FIPBalancedNMeans(n)
    fbn.fit(pop)
    design = gs.Design(pop, num_zones=n, order=gs.Order.from_clusters(pop, fbn.clusters))
    initial_designs.append(design)

In [12]:
criteria = gs.criteria.MoranCriteria()
gbfs = gs.search.GreedyBestFirstSearch(initial_designs, criteria)

In [13]:
gbfs.run(
    max_iterations=1000,
    max_open_set_size=1000,
    top_k=10,
    num_new_order_nodes=5,
    num_new_exchange_nodes=5,
    num_clusters=2,
    num_zones=1,
    num_changes=2,
    num_zone_changes=1,
    random_pull=True,
    exchange_coef=1.0
)

--- Starting Parallel GBFS: Max Iterations=1000, Batch Size=1, Workers=-1 ---
Initial best criteria value: -0.2974
Iter     0/1000 | Best: -0.2974 | Open:    20 | Closed:     0
  [!] New best found at iter 5: -0.2974 (improved by 0.0000)
  [!] New best found at iter 6: -0.2985 (improved by 0.0011)
  [!] New best found at iter 6: -0.2988 (improved by 0.0003)
  [!] New best found at iter 7: -0.2988 (improved by 0.0000)
  [!] New best found at iter 8: -0.2991 (improved by 0.0003)
  [!] New best found at iter 28: -0.2996 (improved by 0.0005)
  [!] New best found at iter 30: -0.2996 (improved by 0.0000)
  [!] New best found at iter 31: -0.2998 (improved by 0.0002)
  [!] New best found at iter 38: -0.3002 (improved by 0.0004)
  [!] New best found at iter 49: -0.3006 (improved by 0.0003)
  [!] New best found at iter 50: -0.3006 (improved by 0.0000)
  [!] New best found at iter 51: -0.3006 (improved by 0.0000)
  [!] New best found at iter 51: -0.3007 (improved by 0.0001)
  [!] New best found a